In [ ]:
%pip install dlt[rest_api] requests

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.stage")

In [ ]:
import dlt
import requests
import pandas as pd
from delta.tables import DeltaTable

In [ ]:
# Frankfurter wraps the official ECB FX data — free, no API key, EUR-based
# Pairs: USD, DKK, GBP, CHF, NOK, SEK (major currencies + Nordic neighbours)
@dlt.resource(name="exchange_rates", write_disposition="merge", primary_key=["date", "currency"])
def fetch_fx_rates():
    response = requests.get(
        "https://api.frankfurter.app/2020-01-01..",
        params={"from": "EUR", "to": "USD,DKK,GBP,CHF,NOK,SEK"},
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()
    for date_str, rates in data["rates"].items():
        for currency, rate in rates.items():
            yield {"date": date_str, "base": data["base"], "currency": currency, "rate": float(rate)}

records = list(fetch_fx_rates())
print(f"Fetched {len(records)} FX rate records")

In [ ]:
spark_df = spark.createDataFrame(pd.DataFrame(records))

tbl = "stocks.stage.fx_exchange_rates_raw"
if not spark.catalog.tableExists(tbl):
    spark_df.write.mode("overwrite").format("delta").saveAsTable(tbl)

DeltaTable.forName(spark, tbl).alias("t").merge(
    spark_df.alias("s"),
    "s.date = t.date AND s.currency = t.currency"
).whenNotMatchedInsertAll().execute()

print(f"stocks.stage.fx_exchange_rates_raw: {spark.table(tbl).count()} rows")